In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:

RAW_MACRO_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../outputs/validation_candidate_features")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw macro directory:", RAW_MACRO_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Raw macro directory: C:\Users\franc\OneDrive\Geop-Model\data\raw
Output directory: C:\Users\franc\OneDrive\Geop-Model\outputs\validation_candidate_features


In [49]:
# Load raw macro data

dgs2 = pd.read_csv(RAW_MACRO_DIR / "DGS2.csv")
epu = pd.read_csv(RAW_MACRO_DIR / "USEPUINDXD.csv")
vix = pd.read_csv(RAW_MACRO_DIR / "VIXCLS.csv")
eurusd = pd.read_csv(RAW_MACRO_DIR / "DEXUSEU.csv")

In [50]:
# Inspect raw macro files

print("DGS2")
display(dgs2.head())
print(dgs2.columns.tolist(), "\n")

print("EPU")
display(epu.head())
print(epu.columns.tolist(), "\n")

print("VIX")
display(vix.head())
print(vix.columns.tolist(), "\n")

print("EUR/USD")
display(eurusd.head())
print(eurusd.columns.tolist())

DGS2


,observation_date,DGS2
0,2017-01-03,1.22
1,2017-01-04,1.24
2,2017-01-05,1.17
3,2017-01-06,1.22
4,2017-01-09,1.21


['observation_date', 'DGS2'] 

EPU


,observation_date,USEPUINDXD
0,2017-01-01,235.10
1,2017-01-02,242.04
2,2017-01-03,89.85
3,2017-01-04,101.99
4,2017-01-05,134.47


['observation_date', 'USEPUINDXD'] 

VIX


,observation_date,VIXCLS
0,2017-01-03,12.85
1,2017-01-04,11.85
2,2017-01-05,11.67
3,2017-01-06,11.32
4,2017-01-09,11.56


['observation_date', 'VIXCLS'] 

EUR/USD


,observation_date,DEXUSEU
0,2017-01-03,1.0416
1,2017-01-04,1.0476
2,2017-01-05,1.0598
3,2017-01-06,1.0560
4,2017-01-09,1.0576


['observation_date', 'DEXUSEU']


In [51]:
# Clean raw macro datasets

def clean_fred_series(df, value_col):
    out = df.copy()
    
    # Standardise date column
    out["observation_date"] = pd.to_datetime(out["observation_date"])
    
    # Convert FRED missing markers / strings to numeric NaN
    out[value_col] = pd.to_numeric(out[value_col], errors="coerce")
    
    # Keep only date + series
    out = out[["observation_date", value_col]]
    
    # Sort chronologically and remove duplicate dates if any
    out = (
        out
        .drop_duplicates(subset="observation_date")
        .sort_values("observation_date")
        .reset_index(drop=True)
    )
    
    return out


dgs2 = clean_fred_series(dgs2, "DGS2")
epu = clean_fred_series(epu, "USEPUINDXD")
vix = clean_fred_series(vix, "VIXCLS")
eurusd = clean_fred_series(eurusd, "DEXUSEU")

In [52]:
# Check cleaned macro series

for name, df_ in {
    "DGS2": dgs2,
    "USEPUINDXD": epu,
    "VIXCLS": vix,
    "DEXUSEU": eurusd
}.items():
    
    print(f"\n{name}")
    print("Shape:", df_.shape)
    print("Date range:",
          df_["observation_date"].min(),
          "to",
          df_["observation_date"].max())
    print("Missing values:")
    print(df_.isna().sum())


DGS2
Shape: (2494, 2)
Date range: 2017-01-03 00:00:00 to 2026-07-24 00:00:00
Missing values:
observation_date      0
DGS2                104
dtype: int64

USEPUINDXD
Shape: (3495, 2)
Date range: 2017-01-01 00:00:00 to 2026-07-27 00:00:00
Missing values:
observation_date    0
USEPUINDXD          0
dtype: int64

VIXCLS
Shape: (2495, 2)
Date range: 2017-01-03 00:00:00 to 2026-07-27 00:00:00
Missing values:
observation_date     0
VIXCLS              60
dtype: int64

DEXUSEU
Shape: (2494, 2)
Date range: 2017-01-03 00:00:00 to 2026-07-24 00:00:00
Missing values:
observation_date      0
DEXUSEU             107
dtype: int64


In [53]:
# Merge raw macro series

macro_raw = (
    eurusd
    .merge(dgs2, on="observation_date", how="outer")
    .merge(epu, on="observation_date", how="outer")
    .merge(vix, on="observation_date", how="outer")
    .sort_values("observation_date")
    .reset_index(drop=True)
)

display(macro_raw.head(10))

print("Shape:", macro_raw.shape)
print("\nDate range:")
print(macro_raw["observation_date"].min(), "to", macro_raw["observation_date"].max())

print("\nMissing values:")
print(macro_raw.isna().sum())

,observation_date,DEXUSEU,DGS2,USEPUINDXD,VIXCLS
0,2017-01-01,NaN,NaN,235.10,NaN
1,2017-01-02,NaN,NaN,242.04,NaN
2,2017-01-03,1.0416,1.22,89.85,12.85
3,2017-01-04,1.0476,1.24,101.99,11.85
4,2017-01-05,1.0598,1.17,134.47,11.67
5,2017-01-06,1.0560,1.22,96.97,11.32
6,2017-01-07,NaN,NaN,110.23,NaN
7,2017-01-08,NaN,NaN,272.01,NaN
8,2017-01-09,1.0576,1.21,121.76,11.56
9,2017-01-10,1.0572,1.19,121.90,11.49


Shape: (3495, 5)

Date range:
2017-01-01 00:00:00 to 2026-07-27 00:00:00

Missing values:
observation_date       0
DEXUSEU             1108
DGS2                1105
USEPUINDXD             0
VIXCLS              1060
dtype: int64


In [54]:
# Restrict to geopolitical research period

START_DATE = "2024-10-01"
END_DATE = "2026-06-11"

macro_raw = macro_raw.loc[
    macro_raw["observation_date"].between(START_DATE, END_DATE)
].copy()

macro_raw = macro_raw.reset_index(drop=True)

print("Shape:", macro_raw.shape)
print(
    "Date range:",
    macro_raw["observation_date"].min(),
    "to",
    macro_raw["observation_date"].max()
)

display(macro_raw.head())
display(macro_raw.tail())

Shape: (619, 5)
Date range: 2024-10-01 00:00:00 to 2026-06-11 00:00:00


,observation_date,DEXUSEU,DGS2,USEPUINDXD,VIXCLS
0,2024-10-01,1.1067,3.61,58.83,19.26
1,2024-10-02,1.1050,3.63,133.57,18.90
2,2024-10-03,1.1015,3.70,76.65,20.49
3,2024-10-04,1.0961,3.93,74.53,19.21
4,2024-10-05,NaN,NaN,93.95,NaN


,observation_date,DEXUSEU,DGS2,USEPUINDXD,VIXCLS
614,2026-06-07,NaN,NaN,338.37,NaN
615,2026-06-08,1.1545,4.15,378.26,18.92
616,2026-06-09,1.1556,4.13,277.00,19.87
617,2026-06-10,1.1550,4.13,186.33,22.22
618,2026-06-11,1.1515,4.05,257.20,19.44


In [55]:
# Paths to LLM annotation outputs

LLM_DATA_DIR = Path("../data/processed")

TRUTHS_FILE = LLM_DATA_DIR / "truths_deepseek_outputs.csv"

truths = pd.read_csv(TRUTHS_FILE)


In [56]:
# Inspect outputs

print("TRUTHS")
print("Shape:", truths.shape)
print("Columns:")
print(truths.columns.tolist())
display(truths.head())


TRUTHS
Shape: (5176, 13)
Columns:
['tweet_index', 'tweet_id', 'date', 'tweet_snippet', 'trade_score', 'sanctions_score', 'fed_pressure_score', 'reasoning', 'source', 'example_ids', 'model', 'prompt_version', 'created_at_utc']


,tweet_index,tweet_id,date,tweet_snippet,trade_score,sanctions_score,fed_pressure_score,reasoning,source,example_ids,model,prompt_version,created_at_utc
0,0,113404826487996526,2024-11-01 00:18:46,Kamala has spent the final week of her failing...,0,0,0,The tweet does not mention anything about the ...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:26:32.881126+00:00
1,1,113404838425751868,2024-11-01 00:21:48,"Just days ago, a young USMC veteran named Nich...",20,0,0,The tweet does not mention the Federal Reserve...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:12.414588+00:00
2,2,113404894982236716,2024-11-01 00:36:11,"GET OUT AND VOTE, NEVADA!!!NEVADA VOTING INFOR...",0,0,0,The tweet does not mention anything related to...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:20.353576+00:00
3,3,113405441290422074,2024-11-01 02:55:07,It was hardworking Patriots like you who built...,0,0,0,The tweet does not mention the Federal Reserve...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:28.870399+00:00
4,4,113405998200625584,2024-11-01 05:16:44,"A GREAT DAY IN NEW MEXICO, NEVADA, AND ARIZONA...",0,0,0,The tweet does not mention anything related to...,llm:native_structured_output,NaN,deepseek-r1:8b,trump-risk-v2.2-grounded-crashsafe,2026-08-21T00:27:35.707895+00:00


In [57]:
# Check likely date columns and missing values

for name, df_ in {
    "Truths": truths
}.items():
    
    print(f"\n{name}")
    print("Missing values:")
    display(df_.isna().sum().sort_values(ascending=False).head(15))


Truths
Missing values:


example_ids           5176
tweet_id                 0
date                     0
tweet_snippet            0
tweet_index              0
trade_score              0
sanctions_score          0
reasoning                0
fed_pressure_score       0
source                   0
model                    0
prompt_version           0
created_at_utc           0
dtype: int64

In [58]:
# Prepare LLM outputs for daily aggregation

SCORE_COLS = [
    "trade_score",
    "sanctions_score",
    "fed_pressure_score"
]

def prepare_llm_scores(df):
    out = df.copy()

    # Convert timestamp to calendar date
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["date"] = out["date"].dt.normalize()

    # Ensure scores are numeric
    for col in SCORE_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


truths = prepare_llm_scores(truths)


In [59]:
for name, df_ in {
    "Truths": truths
}.items():

    print(f"\n{name}")
    print("Rows:", len(df_))
    print(
        "Date range:",
        df_["date"].min(),
        "to",
        df_["date"].max()
    )

    print("\nMissing scores:")
    print(df_[SCORE_COLS].isna().sum())

    print("\nScore ranges:")
    display(df_[SCORE_COLS].agg(["min", "max", "mean"]))


Truths
Rows: 5176
Date range: 2024-11-01 00:00:00 to 2026-06-11 00:00:00

Missing scores:
trade_score           0
sanctions_score       0
fed_pressure_score    0
dtype: int64

Score ranges:


,trade_score,sanctions_score,fed_pressure_score
min,0.000000,0.000000,0.000000
max,100.000000,90.000000,100.000000
mean,7.320131,2.319938,1.146059


In [75]:
def create_daily_geopolitical_features(df):

    daily = (
        df.groupby("date")[SCORE_COLS]
        .agg(["sum", "mean", "max"])
    )

    # Flatten MultiIndex column names
    daily.columns = [
        f"{score.replace('_score', '')}_{stat}"
        for score, stat in daily.columns
    ]

    # Complete daily calendar over the available LLM period
    full_dates = pd.date_range(
        start=daily.index.min(),
        end=daily.index.max(),
        freq="D"
    )

    daily = daily.reindex(full_dates)

    # No-tweet days = zero geopolitical signal
    daily = daily.fillna(0)

    daily.index.name = "date"

    return daily.reset_index()


truths_daily = create_daily_geopolitical_features(truths)


In [76]:
print("DEEPSEEK DAILY")
print("Shape:", truths_daily.shape)
print(
    "Date range:",
    truths_daily["date"].min(),
    "to",
    truths_daily["date"].max()
)
display(truths_daily.head(10))


DEEPSEEK DAILY
Shape: (588, 10)
Date range: 2024-11-01 00:00:00 to 2026-06-11 00:00:00


,date,trade_sum,trade_mean,trade_max,sanctions_sum,sanctions_mean,sanctions_max,fed_pressure_sum,fed_pressure_mean,fed_pressure_max
0,2024-11-01,110.0,4.583333,45.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,2024-11-02,85.0,4.722222,85.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,2024-11-03,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,2024-11-04,0.0,0.000000,0.0,40.0,2.222222,40.0,0.0,0.0,0.0
4,2024-11-05,40.0,3.076923,40.0,0.0,0.000000,0.0,0.0,0.0,0.0
5,2024-11-06,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
6,2024-11-07,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
7,2024-11-08,40.0,20.000000,40.0,30.0,15.000000,30.0,0.0,0.0,0.0
8,2024-11-09,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
9,2024-11-10,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [77]:
# Generate geopolitical candidate features

BASE_GEO_COLS = [
    "trade_sum", "trade_mean", "trade_max",
    "sanctions_sum", "sanctions_mean", "sanctions_max",
    "fed_pressure_sum", "fed_pressure_mean", "fed_pressure_max"
]

GEO_LAGS = [1, 2, 3, 5]
GEO_WINDOWS = [3, 5, 10]


def create_geo_candidate_features(daily_df):
    out = daily_df.copy()
    out = out.sort_values("date").reset_index(drop=True)

    for col in BASE_GEO_COLS:

        # First change
        out[f"{col}_diff"] = out[col].diff()

        # Calendar-day lags
        for lag in GEO_LAGS:
            out[f"{col}_lag{lag}"] = out[col].shift(lag)

        # Moving averages
        for window in GEO_WINDOWS:
            out[f"{col}_ma{window}"] = (
                out[col]
                .rolling(window=window, min_periods=window)
                .mean()
            )

    return out

truths_candidates = create_geo_candidate_features(truths_daily)


In [78]:
# Inspect geopolitical candidate sets

for name, df_ in {
    "Truths": truths_candidates
}.items():

    print(f"\n{name}")
    print("Shape:", df_.shape)
    print("Number of geopolitical predictors:", df_.shape[1] - 1)

    print("\nFirst 20 columns:")
    print(df_.columns[:20].tolist())

    print("\nMissing values caused by transformations:")
    print(df_.isna().sum().sum())


Truths
Shape: (588, 82)
Number of geopolitical predictors: 81

First 20 columns:
['date', 'trade_sum', 'trade_mean', 'trade_max', 'sanctions_sum', 'sanctions_mean', 'sanctions_max', 'fed_pressure_sum', 'fed_pressure_mean', 'fed_pressure_max', 'trade_sum_diff', 'trade_sum_lag1', 'trade_sum_lag2', 'trade_sum_lag3', 'trade_sum_lag5', 'trade_sum_ma3', 'trade_sum_ma5', 'trade_sum_ma10', 'trade_mean_diff', 'trade_mean_lag1']

Missing values caused by transformations:
243


In [79]:
# Create financial-market modelling grid

macro_features = macro_raw.loc[
    macro_raw["DEXUSEU"].notna()
].copy()

macro_features = (
    macro_features
    .sort_values("observation_date")
    .reset_index(drop=True)
)

print("Financial observations:", len(macro_features))
print(
    "Date range:",
    macro_features["observation_date"].min(),
    "to",
    macro_features["observation_date"].max()
)

print("\nMissing values on EUR/USD observation dates:")
print(
    macro_features[
        ["DEXUSEU", "DGS2", "USEPUINDXD", "VIXCLS"]
    ].isna().sum()
)

Financial observations: 424
Date range: 2024-10-01 00:00:00 to 2026-06-11 00:00:00

Missing values on EUR/USD observation dates:
DEXUSEU       0
DGS2          1
USEPUINDXD    0
VIXCLS        2
dtype: int64


In [80]:
# Keep complete financial-market observations

MACRO_RAW_COLS = [
    "DEXUSEU",
    "DGS2",
    "USEPUINDXD",
    "VIXCLS"
]

macro_features = (
    macro_features
    .dropna(subset=MACRO_RAW_COLS)
    .copy()
    .reset_index(drop=True)
)

print("Complete financial observations:", len(macro_features))

print(
    "Date range:",
    macro_features["observation_date"].min(),
    "to",
    macro_features["observation_date"].max()
)

print("\nMissing values:")
print(macro_features[MACRO_RAW_COLS].isna().sum())

Complete financial observations: 422
Date range: 2024-10-01 00:00:00 to 2026-06-11 00:00:00

Missing values:
DEXUSEU       0
DGS2          0
USEPUINDXD    0
VIXCLS        0
dtype: int64


In [81]:
# EUR/USD target

macro_features["DEXUSEU_logreturn"] = (
    np.log(macro_features["DEXUSEU"])
    .diff()
)

In [82]:
# Generate macro candidate features

MACRO_BASE_COLS = [
    "DGS2",
    "USEPUINDXD",
    "VIXCLS"
]

MACRO_LAGS = [1, 2, 3, 5]
MACRO_WINDOWS = [3, 5, 10]


for col in MACRO_BASE_COLS:

    # Current change
    macro_features[f"{col}_diff"] = (
        macro_features[col].diff()
    )

    # Lagged levels
    for lag in MACRO_LAGS:
        macro_features[f"{col}_lag{lag}"] = (
            macro_features[col].shift(lag)
        )

    # Lagged changes
    for lag in MACRO_LAGS:
        macro_features[f"{col}_diff_lag{lag}"] = (
            macro_features[f"{col}_diff"].shift(lag)
        )

    # Moving averages of levels
    for window in MACRO_WINDOWS:
        macro_features[f"{col}_ma{window}"] = (
            macro_features[col]
            .rolling(
                window=window,
                min_periods=window
            )
            .mean()
        )

In [83]:
# Lagged EUR/USD returns

for lag in MACRO_LAGS:
    macro_features[f"DEXUSEU_logreturn_lag{lag}"] = (
        macro_features["DEXUSEU_logreturn"].shift(lag)
    )

In [84]:
# Check macro candidate dataset

print("Shape:", macro_features.shape)

print("\nMissing values after feature construction:")
print(
    macro_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nFirst date with complete candidate information:")

complete_macro = macro_features.dropna()

print(complete_macro["observation_date"].min())
print("Complete rows:", len(complete_macro))

Shape: (422, 46)

Missing values after feature construction:
USEPUINDXD_ma10           9
DGS2_ma10                 9
VIXCLS_ma10               9
USEPUINDXD_diff_lag5      6
DEXUSEU_logreturn_lag5    6
VIXCLS_diff_lag5          6
DGS2_diff_lag5            6
USEPUINDXD_lag5           5
DGS2_lag5                 5
VIXCLS_lag5               5
VIXCLS_ma5                4
VIXCLS_diff_lag3          4
DEXUSEU_logreturn_lag3    4
DGS2_diff_lag3            4
DGS2_ma5                  4
USEPUINDXD_diff_lag3      4
USEPUINDXD_ma5            4
VIXCLS_lag3               3
VIXCLS_diff_lag2          3
DEXUSEU_logreturn_lag2    3
dtype: int64

First date with complete candidate information:
2024-10-15 00:00:00
Complete rows: 413


In [85]:
# Rename geopolitical date column to match macro data

truths_candidates = truths_candidates.rename(
    columns={"date": "observation_date"}
)

print(truths_candidates.columns[:5])
print(truths_candidates["observation_date"].min(),
      truths_candidates["observation_date"].max())

Index(['observation_date', 'trade_sum', 'trade_mean', 'trade_max',
       'sanctions_sum'],
      dtype='str')
2024-11-01 00:00:00 2026-06-11 00:00:00


In [86]:
# Merge macro and geopolitical candidate features

truths_full = macro_features.merge(
    truths_candidates,
    on="observation_date",
    how="left"
)

print("Truths:", truths_full.shape)


Truths: (422, 127)


In [87]:
# Check merged datasets

for name, df_ in {
    "Truths": truths_full
}.items():

    print(f"\n{name}")
    print("Shape:", df_.shape)

    print(
        "Date range:",
        df_["observation_date"].min(),
        "to",
        df_["observation_date"].max()
    )

    print("Total missing values:", df_.isna().sum().sum())

    print("\nColumns with most missing values:")
    print(
        df_.isna()
        .sum()
        .sort_values(ascending=False)
        .head(15)
    )


Truths
Shape: (422, 127)
Date range: 2024-10-01 00:00:00 to 2026-06-11 00:00:00
Total missing values: 2068

Columns with most missing values:
trade_sum_ma10            28
fed_pressure_max_ma10     28
fed_pressure_mean_ma10    28
sanctions_sum_ma10        28
sanctions_mean_ma10       28
trade_max_ma10            28
trade_mean_ma10           28
sanctions_max_ma10        28
fed_pressure_sum_ma10     28
trade_sum_lag5            25
sanctions_sum_lag5        25
fed_pressure_mean_lag5    25
fed_pressure_max_lag5     25
fed_pressure_sum_lag5     25
sanctions_max_lag5        25
dtype: int64


In [88]:
missing_geo_dates = truths_full.loc[
    truths_full["trade_sum"].isna(),
    "observation_date"
]

print("Dates with no geopolitical observation:")
print(missing_geo_dates)

Dates with no geopolitical observation:
0    2024-10-01
1    2024-10-02
2    2024-10-03
3    2024-10-04
4    2024-10-07
5    2024-10-08
6    2024-10-09
7    2024-10-10
8    2024-10-11
9    2024-10-15
10   2024-10-16
11   2024-10-17
12   2024-10-18
13   2024-10-21
14   2024-10-22
15   2024-10-23
16   2024-10-24
17   2024-10-25
18   2024-10-28
19   2024-10-29
20   2024-10-30
21   2024-10-31
Name: observation_date, dtype: datetime64[us]


In [89]:
print("Raw Truths first timestamp:", truths["date"].min())
print("Raw Truths last timestamp:", truths["date"].max())

display(
    truths.sort_values("date")[
        ["date", "trade_score", "sanctions_score", "fed_pressure_score"]
    ].head(20)
)

Raw Truths first timestamp: 2024-11-01 00:00:00
Raw Truths last timestamp: 2026-06-11 00:00:00


,date,trade_score,sanctions_score,fed_pressure_score
0,2024-11-01,0,0,0
23,2024-11-01,0,0,0
22,2024-11-01,0,0,0
21,2024-11-01,0,0,0
20,2024-11-01,0,0,0
19,2024-11-01,0,0,0
18,2024-11-01,5,0,0
17,2024-11-01,45,0,0
15,2024-11-01,0,0,0
14,2024-11-01,0,0,0


In [90]:
# Define target

TARGET = "DEXUSEU_logreturn"

# Columns that are not candidate predictors
NON_PREDICTORS = [
    "observation_date",
    "DEXUSEU",              # contemporaneous EUR/USD level
    TARGET                   # target itself
]

In [91]:
# Find common complete modelling dates

truths_complete_dates = truths_full.dropna()["observation_date"]

common_dates = (
    pd.Index(truths_complete_dates)
    .sort_values()
)

print("Truths complete dates:", len(truths_complete_dates))
print("Common complete dates:", len(common_dates))

print(
    "Common date range:",
    common_dates.min(),
    "to",
    common_dates.max()
)

Truths complete dates: 394
Common complete dates: 394
Common date range: 2024-11-12 00:00:00 to 2026-06-11 00:00:00


In [92]:
# Restrict Truths model to identical sample

truths_model = (
    truths_full[
        truths_full["observation_date"].isin(common_dates)
    ]
    .copy()
    .reset_index(drop=True)
)


print("Truths:", truths_model.shape)

print("\nMissing values:")
print("Truths:", truths_model.isna().sum().sum())



Truths: (394, 127)

Missing values:
Truths: 0


In [93]:
# Candidate predictor matrices

predictor_cols = [
    col for col in truths_model.columns
    if col not in NON_PREDICTORS
]

X_truths = truths_model[predictor_cols].copy()

y_truths = truths_model[TARGET].copy()

print("Truths X:", X_truths.shape)

print("Truths y:", y_truths.shape)

print("\nNumber of candidate predictors:", len(predictor_cols))

Truths X: (394, 124)
Truths y: (394,)

Number of candidate predictors: 124


In [94]:
# Check predictors with zero variance

constant_truths = X_truths.columns[
    X_truths.nunique() <= 1
].tolist()

print("Constant predictors:")
print(constant_truths)

Constant predictors:
[]


In [95]:
# Find exact duplicate columns

def find_duplicate_columns(df):
    duplicates = []

    cols = df.columns

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if df[cols[i]].equals(df[cols[j]]):
                duplicates.append((cols[i], cols[j]))

    return duplicates


duplicate_truths = find_duplicate_columns(X_truths)

print("Exact duplicates:")
print(duplicate_truths)

Exact duplicates:
[]


In [96]:
# Find highly correlated predictor pairs

def high_correlation_pairs(df, threshold=0.995):
    corr = df.corr().abs()

    upper = corr.where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )

    pairs = []

    for col in upper.columns:
        for row in upper.index:
            value = upper.loc[row, col]

            if pd.notna(value) and value >= threshold:
                pairs.append((row, col, value))

    return sorted(
        pairs,
        key=lambda x: x[2],
        reverse=True
    )


high_corr_truths = high_correlation_pairs(
    X_truths,
    threshold=0.995
)

print("Truths pairs >= 0.995:")
for pair in high_corr_truths:
    print(pair)

Truths pairs >= 0.995:
('DGS2_ma3', 'DGS2_ma5', np.float64(0.9960873772022556))
('DGS2_lag1', 'DGS2_ma3', np.float64(0.9954981020080792))


In [97]:
# Final candidate feature datasets

final_cols = [
    "observation_date",
    TARGET
] + predictor_cols

candidate_truths = truths_model[final_cols].copy()

print("Truths final shape:", candidate_truths.shape)

print("\nExpected:")
print("394 observations")
print("126 columns = date + target + 124 predictors")

print("\nMissing values:")
print("Truths:", candidate_truths.isna().sum().sum())


Truths final shape: (394, 126)

Expected:
394 observations
126 columns = date + target + 124 predictors

Missing values:
Truths: 0


In [98]:
# Save final candidate datasets

truths_path = OUTPUT_DIR / "validation_candidate_features.csv"

candidate_truths.to_csv(
    truths_path,
    index=False
)

print("Saved:")
print(truths_path)

Saved:
..\outputs\validation_candidate_features\validation_candidate_features.csv


In [99]:
# Create feature manifest

def classify_feature(col):

    if col.startswith("DEXUSEU_logreturn_lag"):
        source = "EURUSD"
    elif col.startswith(("DGS2", "USEPUINDXD", "VIXCLS")):
        source = "Macro"
    else:
        source = "Geopolitical"

    if "_diff_lag" in col:
        transformation = "lagged_difference"
    elif col.endswith("_diff"):
        transformation = "difference"
    elif "_lag" in col:
        transformation = "lag"
    elif "_ma" in col:
        transformation = "moving_average"
    else:
        transformation = "level_or_daily_aggregation"

    return source, transformation


manifest_rows = []

for col in predictor_cols:

    source, transformation = classify_feature(col)

    manifest_rows.append({
        "feature": col,
        "source": source,
        "transformation": transformation
    })


feature_manifest = pd.DataFrame(manifest_rows)

display(feature_manifest)

print("\nFeatures by source:")
print(feature_manifest["source"].value_counts())

print("\nFeatures by transformation:")
print(feature_manifest["transformation"].value_counts())

,feature,source,transformation
0,DGS2,Macro,level_or_daily_aggregation
1,USEPUINDXD,Macro,level_or_daily_aggregation
2,VIXCLS,Macro,level_or_daily_aggregation
3,DGS2_diff,Macro,difference
4,DGS2_lag1,Macro,lag
...,...,...,...
119,fed_pressure_max_lag3,Geopolitical,lag
120,fed_pressure_max_lag5,Geopolitical,lag
121,fed_pressure_max_ma3,Geopolitical,moving_average
122,fed_pressure_max_ma5,Geopolitical,moving_average



Features by source:
source
Geopolitical    81
Macro           39
EURUSD           4
Name: count, dtype: int64

Features by transformation:
transformation
lag                           52
moving_average                39
difference                    12
lagged_difference             12
level_or_daily_aggregation     9
Name: count, dtype: int64


In [100]:
# Cell 36 — Save feature manifest

manifest_path = OUTPUT_DIR / "validation_candidate_feature_manifest.csv"

feature_manifest.to_csv(
    manifest_path,
    index=False
)

print("Saved:", manifest_path)

Saved: ..\outputs\validation_candidate_features\validation_candidate_feature_manifest.csv


In [102]:
# Final sanity check

print("=" * 50)
print("CANDIDATE FEATURE DATASET COMPLETE")
print("=" * 50)

print(f"Observations: {len(candidate_truths)}")
print(f"Candidate predictors: {len(predictor_cols)}")

print(
    "Sample:",
    candidate_truths["observation_date"].min().date(),
    "to",
    candidate_truths["observation_date"].max().date()
)

print("\nTarget:")
print(TARGET)

print("\nTruths dataset:")
print(candidate_truths.shape)

print("\nFeature groups:")
print(feature_manifest["source"].value_counts())

print("\nMissing values:")
print("Truths:", candidate_truths.isna().sum().sum())

CANDIDATE FEATURE DATASET COMPLETE
Observations: 394
Candidate predictors: 124
Sample: 2024-11-12 to 2026-06-11

Target:
DEXUSEU_logreturn

Truths dataset:
(394, 126)

Feature groups:
source
Geopolitical    81
Macro           39
EURUSD           4
Name: count, dtype: int64

Missing values:
Truths: 0
